# Ackermann Function & Computability Analyzer

This interactive notebook explores the **Ackermann–Péter function**, one of computer science's hallmark demonstrations of a **total computable function** that falls outside the class of **primitive recursive** functions. By instrumenting every recursive step, we can directly observe how quickly stack depth, call volume, and runtime escalate.

## Mathematical Definition

Given two non-negative integers $m$ and $n$, the two-argument Ackermann–Péter function $A(m, n)$ is specified by three rules:

$$
A(m, n) = \begin{cases}
n + 1 & \text{if } m = 0 \\
A(m - 1, 1) & \text{if } m > 0 \text{ and } n = 0 \\
A(m - 1, A(m, n - 1)) & \text{if } m > 0 \text{ and } n > 0
\end{cases}
$$

### Why Is It Special?
* **Hyper-Exponential Scaling:** Conventional fast-growing functions such as $2^n$ or $n!$ are eventually overtaken by $A(m, n)$, which outpaces every primitive recursive operation—from repeated addition all the way up to fixed-height power towers.
* **Foundations of Computability:** Wilhelm Ackermann and Rózsa Péter showed that general computation (unbounded recursion or `while`-style iteration) is strictly more expressive than bounded-loop programming (`for` loops whose iteration counts are fixed in advance).

## Our Implementation Blueprint

The notebook is organized into four hands-on components:
1. **Instrumented Recursion Engine:** Evaluates $A(m, n)$ while recording wall-clock duration, total function invocations, and maximum call-stack depth, validated with test cases including $A(2, 5)$, $A(3, 4)$, $A(3, 5)$, and the edge input $A(4, 0)$.
2. **Parameter Sweep & Derived Metrics Table:** Collects call counts, stack depth, runtime (ms), per-call overhead (ns/call), and amplification ratios across $m \le 3, n \le 6$ inside a `pandas` DataFrame.
3. **Visual Growth & Benchmarking Suite:** Plots 2×2 complexity metrics and runs multi-trial latency/throughput benchmarks comparing **naive recursion** against **memoized recursion** with error bars and speedup charts.
4. **Interactive Colab Dashboard:** Provides `ipywidgets` controls (initialized at $A(3, 4)$ and supporting safe $m = 4, n = 0$ evaluation) with a step-by-step recursion breakdown.


## 1. Core Ackermann Engine with Metrics Tracking

To observe resource consumption without altering the mathematical signature of the recursive helper, we pass a lightweight `AckermannMetrics` tracker alongside $(m, n)$ and wrap the execution inside `evaluate_ackermann(m, n)`.

After defining the engine, an automated verification block checks six representative cases—including `A(2, 5)`, `A(3, 4)`, `A(3, 5)`, and the safe `m = 4` boundary `A(4, 0)`—before moving on to the full grid.


In [ ]:
import sys
import time

# Raise Python's recursion limit so deeper safe cases like A(3, 6) (depth 511) complete cleanly
sys.setrecursionlimit(60000)


class AckermannMetrics:
    def __init__(self):
        self.calls = 0
        self.max_depth = 0
        self.current_depth = 0


def _ack_recursive(m, n, metrics):
    metrics.calls += 1
    metrics.current_depth += 1
    if metrics.current_depth > metrics.max_depth:
        metrics.max_depth = metrics.current_depth

    # Evaluate according to the three Ackermann-Péter cases
    if m == 0:
        result = n + 1
    elif m > 0 and n == 0:
        result = _ack_recursive(m - 1, 1, metrics)
    else:
        inner_result = _ack_recursive(m, n - 1, metrics)
        result = _ack_recursive(m - 1, inner_result, metrics)

    metrics.current_depth -= 1
    return result


def evaluate_ackermann(m, n):
    """
    Evaluate A(m, n) and collect call count, peak recursion depth, and runtime.
    Returns (result, metrics_dict).
    """
    if m < 0 or n < 0:
        raise ValueError("Both m and n must be non-negative integers.")

    metrics = AckermannMetrics()
    start_time = time.perf_counter()

    try:
        result = _ack_recursive(m, n, metrics)
    except RecursionError:
        result = "Recursion Depth Exceeded"

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    return result, {
        "m": m,
        "n": n,
        "result": result,
        "calls": metrics.calls,
        "max_depth": metrics.max_depth,
        "time_seconds": elapsed_time,
    }


# Verification suite across distinct parameter regimes
test_examples = [
    (0, 7, 8),        # Case m=0: n + 1
    (1, 6, 8),        # Case m=1: n + 2
    (2, 5, 13),       # Case m=2: 2n + 3
    (3, 4, 125),      # Case m=3: 2^(n+3) - 3
    (3, 5, 253),      # Deeper m=3 benchmark
    (4, 0, 13),       # Safe m=4 boundary: A(4, 0) reduces to A(3, 1) = 13
]

print("Running Ackermann Verification Tests:")
for m_in, n_in, expected in test_examples:
    res, stats = evaluate_ackermann(m_in, n_in)
    assert res == expected, f"Expected A({m_in}, {n_in}) = {expected}, got {res}"
    print(
        f"  PASS A({m_in}, {n_in}) = {res:<4} | "
        f"calls={stats['calls']:<6} | max_depth={stats['max_depth']:<4} | "
        f"time={stats['time_seconds']:.6f}s"
    )


## 2. Tabular Growth & Computational Metrics Evaluator

Next, we benchmark every coordinate pair across an expanded rectangular grid:
* $m \in \{0, 1, 2, 3\}$
* $n \in \{0, 1, 2, 3, 4, 5, 6\}$

In addition to raw result, call volume, stack depth, and runtime, we compute derived efficiency metrics (`time_ms`, `ns_per_call`, and `calls_per_result`) to quantify how much redundant work pure recursion performs.


In [ ]:
import pandas as pd

# Collect metrics across all safe (m, n) combinations
data_records = []

for m in range(4):
    for n in range(7):
        # Omit m >= 4 from the grid sweep because A(4, 1) = 65,533 exhausts stack and time budgets
        _, stats = evaluate_ackermann(m, n)
        stats["time_ms"] = stats["time_seconds"] * 1_000.0
        stats["ns_per_call"] = (stats["time_seconds"] * 1e9) / stats["calls"]
        stats["calls_per_result"] = stats["calls"] / stats["result"]
        data_records.append(stats)

# Assemble results into a formatted DataFrame
df_metrics = pd.DataFrame(data_records)
pd.options.display.float_format = "{:,.4f}".format

display(df_metrics)


## 3. Visual Growth, Complexity & Benchmarking Analysis

### 3.1 Grid Complexity Metrics ($m \le 3, n \le 6$)
This 2×2 metric dashboard visualizes four complementary dimensions across the input grid:
1. **Recursive Calls (Log Scale)** — separates linear, quadratic, and exponential call regimes.
2. **Peak Stack Depth** — tracks maximum simultaneous call frames held in memory.
3. **Execution Time (ms)** — highlights the sudden wall-clock spike at $m = 3$.
4. **Call Amplification (`calls / result`)** — shows how many recursive steps are spent per unit of output value.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Recursive Call Count (Log Scale)
sns.lineplot(
    ax=axes[0, 0],
    data=df_metrics,
    x="n",
    y="calls",
    hue="m",
    marker="o",
    palette="viridis",
    linewidth=2.5,
)
axes[0, 0].set_title("1. Recursive Calls Growth (Log Scale)", fontsize=13, weight="bold")
axes[0, 0].set_xlabel("Input (n)", fontsize=11)
axes[0, 0].set_ylabel("Total Recursive Calls", fontsize=11)
axes[0, 0].set_yscale("log")
axes[0, 0].set_xticks(range(7))

# 2. Peak Recursion Stack Depth
sns.lineplot(
    ax=axes[0, 1],
    data=df_metrics,
    x="n",
    y="max_depth",
    hue="m",
    marker="D",
    palette="crest",
    linewidth=2.5,
)
axes[0, 1].set_title("2. Peak Stack Depth Growth (Log Scale)", fontsize=13, weight="bold")
axes[0, 1].set_xlabel("Input (n)", fontsize=11)
axes[0, 1].set_ylabel("Max Stack Depth (Frames)", fontsize=11)
axes[0, 1].set_yscale("log")
axes[0, 1].set_xticks(range(7))

# 3. Execution Time (ms)
sns.lineplot(
    ax=axes[1, 0],
    data=df_metrics,
    x="n",
    y="time_ms",
    hue="m",
    marker="s",
    palette="magma",
    linewidth=2.5,
)
axes[1, 0].set_title("3. Wall-Clock Execution Time (ms)", fontsize=13, weight="bold")
axes[1, 0].set_xlabel("Input (n)", fontsize=11)
axes[1, 0].set_ylabel("Duration (milliseconds)", fontsize=11)
axes[1, 0].set_xticks(range(7))

# 4. Call Amplification Ratio (calls per unit of output)
sns.lineplot(
    ax=axes[1, 1],
    data=df_metrics,
    x="n",
    y="calls_per_result",
    hue="m",
    marker="^",
    palette="flare",
    linewidth=2.5,
)
axes[1, 1].set_title("4. Call Amplification (Calls / Output Value)", fontsize=13, weight="bold")
axes[1, 1].set_xlabel("Input (n)", fontsize=11)
axes[1, 1].set_ylabel("Recursive Calls per Result Unit", fontsize=11)
axes[1, 1].set_yscale("log")
axes[1, 1].set_xticks(range(7))

plt.suptitle(
    "Ackermann Function Complexity & Resource Metrics (m <= 3, n <= 6)",
    fontsize=16,
    weight="bold",
)
plt.tight_layout()
plt.show()


### 3.2 Multi-Trial Benchmarking: Naive Recursion vs. Memoized Recursion

Single-run timings on microsecond workloads suffer from OS scheduling noise. Below, we run a **multi-trial benchmark** across `A(3, 0)` through `A(3, 6)` plus `A(4, 0)` comparing:
* **Naive Recursion:** Recomputes overlapping subproblems from scratch every time.
* **Memoized Recursion:** Caches `(m, n)` sub-results in a dictionary while preserving identical mathematical outputs.

We record **mean/std/min/max latency (ms)**, **total call reduction (%)**, **throughput (calls/sec)**, and **speedup factor**, then plot all four benchmark metrics.


In [ ]:
import statistics
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


def _ack_memoized(m, n, cache, metrics):
    metrics.calls += 1
    metrics.current_depth += 1
    if metrics.current_depth > metrics.max_depth:
        metrics.max_depth = metrics.current_depth

    if (m, n) in cache:
        result = cache[(m, n)]
    else:
        if m == 0:
            result = n + 1
        elif m > 0 and n == 0:
            result = _ack_memoized(m - 1, 1, cache, metrics)
        else:
            inner = _ack_memoized(m, n - 1, cache, metrics)
            result = _ack_memoized(m - 1, inner, cache, metrics)
        cache[(m, n)] = result

    metrics.current_depth -= 1
    return result


def evaluate_ackermann_memoized(m, n):
    metrics = AckermannMetrics()
    cache = {}
    start_time = time.perf_counter()
    result = _ack_memoized(m, n, cache, metrics)
    elapsed_time = time.perf_counter() - start_time
    return result, {
        "calls": metrics.calls,
        "max_depth": metrics.max_depth,
        "unique_states": len(cache),
        "time_seconds": elapsed_time,
    }


benchmark_inputs = [
    (3, 0, 30),
    (3, 1, 30),
    (3, 2, 30),
    (3, 3, 25),
    (3, 4, 20),
    (3, 5, 10),
    (3, 6, 5),
    (4, 0, 30),
]

bench_rows = []
for m_val, n_val, trials in benchmark_inputs:
    naive_ms, memo_ms = [], []
    naive_calls = memo_calls = naive_depth = memo_depth = unique_states = 0
    expected_res = None

    # Warm-up pass
    r_w, _ = evaluate_ackermann(m_val, n_val)
    r_m, _ = evaluate_ackermann_memoized(m_val, n_val)
    assert r_w == r_m

    for _ in range(trials):
        r1, s1 = evaluate_ackermann(m_val, n_val)
        r2, s2 = evaluate_ackermann_memoized(m_val, n_val)
        assert r1 == r2
        expected_res = r1
        naive_calls, naive_depth = s1["calls"], s1["max_depth"]
        memo_calls, memo_depth, unique_states = s2["calls"], s2["max_depth"], s2["unique_states"]
        naive_ms.append(s1["time_seconds"] * 1000.0)
        memo_ms.append(s2["time_seconds"] * 1000.0)

    n_mean = statistics.mean(naive_ms)
    n_std = statistics.stdev(naive_ms) if trials > 1 else 0.0
    m_mean = statistics.mean(memo_ms)
    m_std = statistics.stdev(memo_ms) if trials > 1 else 0.0
    naive_throughput_k = (naive_calls / (n_mean / 1000.0)) / 1000.0 if n_mean > 0 else 0.0
    ns_per_call = (n_mean * 1e6) / naive_calls if naive_calls > 0 else 0.0

    bench_rows.append(
        {
            "input": f"A({m_val},{n_val})",
            "trials": trials,
            "result": expected_res,
            "naive_calls": naive_calls,
            "memo_calls": memo_calls,
            "call_reduction_pct": (1.0 - memo_calls / naive_calls) * 100.0,
            "unique_states": unique_states,
            "naive_depth": naive_depth,
            "memo_depth": memo_depth,
            "naive_mean_ms": n_mean,
            "naive_std_ms": n_std,
            "naive_min_ms": min(naive_ms),
            "naive_max_ms": max(naive_ms),
            "memo_mean_ms": m_mean,
            "memo_std_ms": m_std,
            "naive_kcalls_per_s": naive_throughput_k,
            "naive_ns_per_call": ns_per_call,
            "speedup_x": n_mean / m_mean if m_mean > 0 else 0.0,
        }
    )

df_bench = pd.DataFrame(bench_rows)
display(df_bench)

# Plot 2x2 Benchmark Metric Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
x = np.arange(len(df_bench))
width = 0.36
labels = df_bench["input"].tolist()

# Panel 1: Naive vs Memoized Call Volume (Log Scale)
b1 = axes[0, 0].bar(x - width / 2, df_bench["naive_calls"], width, label="Naive Recursion", color="#4c72b0")
b2 = axes[0, 0].bar(x + width / 2, df_bench["memo_calls"], width, label="Memoized", color="#55a868")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("1. Call Volume: Naive vs. Memoized (Log Scale)", fontsize=13, weight="bold")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(labels)
axes[0, 0].set_ylabel("Total Function Calls")
axes[0, 0].legend()
axes[0, 0].grid(axis="y", linestyle="--", alpha=0.5)
for bar in b1:
    h = bar.get_height()
    axes[0, 0].annotate(f"{h:,}", (bar.get_x() + bar.get_width() / 2, h),
                        ha="center", va="bottom", fontsize=8, xytext=(0, 2), textcoords="offset points")

# Panel 2: Multi-Trial Latency Mean ± Std (Log Scale)
axes[0, 1].errorbar(x, df_bench["naive_mean_ms"], yerr=df_bench["naive_std_ms"],
                    fmt="-o", capsize=4, linewidth=2.2, label="Naive (mean ± std)", color="#c44e52")
axes[0, 1].errorbar(x, df_bench["memo_mean_ms"], yerr=df_bench["memo_std_ms"],
                    fmt="-s", capsize=4, linewidth=2.2, label="Memoized (mean ± std)", color="#55a868")
axes[0, 1].set_yscale("log")
axes[0, 1].set_title("2. Multi-Trial Latency with Std Dev (Log ms)", fontsize=13, weight="bold")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(labels)
axes[0, 1].set_ylabel("Execution Latency (ms)")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

# Panel 3: Memoization Speedup Factor (x)
bars_sp = axes[1, 0].bar(x, df_bench["speedup_x"], color="#8172b3", width=0.55)
axes[1, 0].axhline(1.0, color="#666666", linestyle="--", linewidth=1.2, label="1.0x Parity")
axes[1, 0].set_title("3. Memoization Speedup Factor (Naive / Memoized)", fontsize=13, weight="bold")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(labels)
axes[1, 0].set_ylabel("Speedup (x faster)")
axes[1, 0].legend()
axes[1, 0].grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars_sp:
    h = bar.get_height()
    axes[1, 0].annotate(f"{h:.1f}x", (bar.get_x() + bar.get_width() / 2, h),
                        ha="center", va="bottom", fontsize=9, weight="bold", xytext=(0, 3), textcoords="offset points")

# Panel 4: Engine Throughput (kCalls/sec) & Per-Call Overhead (ns/call)
bars_tp = axes[1, 1].bar(x, df_bench["naive_kcalls_per_s"], color="#dd8452", width=0.55, label="Throughput (kCalls/s)")
axes[1, 1].set_title("4. Naive Call Throughput & Per-Call Overhead", fontsize=13, weight="bold")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_ylabel("Throughput (Thousand Calls / sec)")
axes[1, 1].grid(axis="y", linestyle="--", alpha=0.5)
ax_ns = axes[1, 1].twinx()
ax_ns.plot(x, df_bench["naive_ns_per_call"], color="#2b5c8f", marker="D", linewidth=2.0, label="Overhead (ns/call)")
ax_ns.set_ylabel("Per-Call Cost (ns/call)", color="#2b5c8f")
axes[1, 1].set_ylim(0, df_bench["naive_kcalls_per_s"].max() * 1.22)
ax_ns.set_ylim(df_bench["naive_ns_per_call"].min() * 0.92, df_bench["naive_ns_per_call"].max() * 1.18)
lines1, labels1 = axes[1, 1].get_legend_handles_labels()
lines2, labels2 = ax_ns.get_legend_handles_labels()
axes[1, 1].legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.suptitle("Ackermann Benchmarking Suite: Latency, Throughput & Memoization Speedup", fontsize=16, weight="bold")
plt.tight_layout()
plt.show()


## 4. Interactive Analyzer Dashboard

Use the `ipywidgets` control panel below to experiment with individual $(m, n)$ pairs. The sliders start at `m = 3, n = 4` (with `m` ranging up to `4` and `n` up to `6`). You can inspect safe boundary behavior at `A(4, 0)`, while a built-in safety guard intercepts `m = 4, n >= 1` before it can stall the Colab kernel.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Interactive parameter sliders and trigger button
m_slider = widgets.IntSlider(value=3, min=0, max=4, step=1, description="Value of m:")
n_slider = widgets.IntSlider(value=4, min=0, max=6, step=1, description="Value of n:")
calculate_btn = widgets.Button(
    description="Calculate & Analyze",
    button_style="primary",
    icon="calculator",
)
output_area = widgets.Output()


def generate_trace_explanation(m, n):
    """
    Produce a readable, theme-aware HTML summary describing the initial
    recursive transition taken for A(m, n).
    """
    explanation = [
        f"<h3 style='color: var(--colab-primary-text-color, #212529); margin-top: 15px;'>"
        f"Initial Step Breakdown for A({m}, {n}):</h3>"
    ]
    if m == 0:
        explanation.append(
            "<p style='color: var(--colab-primary-text-color, #333333);'>"
            "- Because <b>m = 0</b>, recursion terminates immediately via Case 1: <i>A(0, n) = n + 1</i>.</p>"
        )
        explanation.append(
            f"<p style='color: var(--colab-primary-text-color, #333333);'>"
            f"- <b>Direct Output:</b> A(0, {n}) = {n} + 1 = {n + 1}.</p>"
        )
    elif m > 0 and n == 0:
        explanation.append(
            f"<p style='color: var(--colab-primary-text-color, #333333);'>"
            f"- Because <b>m = {m} &gt; 0</b> and <b>n = 0</b>, Case 2 applies: <i>A(m, 0) = A(m - 1, 1)</i>.</p>"
        )
        explanation.append(
            f"<p style='color: var(--colab-primary-text-color, #333333);'>"
            f"- Evaluating A({m}, 0) therefore steps down directly to <b>A({m - 1}, 1)</b>.</p>"
        )
    else:
        explanation.append(
            f"<p style='color: var(--colab-primary-text-color, #333333);'>"
            f"- Because both <b>m = {m} &gt; 0</b> and <b>n = {n} &gt; 0</b>, Case 3 triggers nested recursion: "
            f"<i>A(m, n) = A(m - 1, A(m, n - 1))</i>.</p>"
        )
        explanation.append(
            f"<p style='color: var(--colab-primary-text-color, #333333);'>"
            f"- Step 1: Resolve the inner subproblem <b>A({m}, {n - 1})</b>.</p>"
        )
        explanation.append(
            f"<p style='color: var(--colab-primary-text-color, #333333);'>"
            f"- Step 2: Feed that returned value as the second argument into <b>A({m - 1}, ...)</b>.</p>"
        )
    return "\n".join(explanation)


def on_click_calculate(b):
    with output_area:
        clear_output()
        m_val = m_slider.value
        n_val = n_slider.value

        # Protect the notebook runtime against intractable m=4, n>=1 calls
        if m_val == 4 and n_val > 0:
            warning_html = f"""
            <div style="padding: 15px; border-radius: 8px; background-color: #fff3cd; border: 1px solid #ffeeba; color: #856404;">
                <h3 style="margin-top: 0;">Execution Halted by Safety Guard</h3>
                <p><b>A({m_val}, {n_val})</b> cannot be evaluated safely using direct recursion.</p>
                <p>Even <i>A(4, 1) = 65,533</i> exceeds practical stack and runtime limits, and <i>A(4, 2)</i> spans 19,729 digits.
                Keep <b>n = 0</b> when <i>m = 4</i>, or select <b>m &le; 3</b>.</p>
            </div>
            """
            display(HTML(warning_html))
            return

        if m_val == 3 and n_val >= 5:
            print(f"Note: Running deep safe input A({m_val}, {n_val})...")

        res, stats = evaluate_ackermann(m_val, n_val)

        metrics_html = f"""
        <div style="padding: 15px; border-radius: 8px; background-color: var(--colab-primary-surface-color, #f8f9fa); border: 1px solid var(--colab-border-color, #dee2e6);">
            <h3 style="margin-top: 0; color: #007bff;">Evaluation Complete</h3>
            <p style="font-size: 16px; color: var(--colab-primary-text-color, #212529);">
                <b>Computed Value:</b>
                <span style="color: #dc3545; font-weight: bold;">A({m_val}, {n_val}) = {res}</span>
            </p>
            <table style="width: 100%; border-collapse: collapse; margin-top: 10px; color: var(--colab-primary-text-color, #212529);">
                <tr style="border-bottom: 2px solid var(--colab-border-color, #ccc);">
                    <th style="text-align: left; padding: 8px; color: var(--colab-primary-text-color, #212529);">Metric</th>
                    <th style="text-align: left; padding: 8px; color: var(--colab-primary-text-color, #212529);">Measured Output</th>
                </tr>
                <tr style="border-bottom: 1px solid var(--colab-border-color, #ddd);">
                    <td style="padding: 8px; color: var(--colab-secondary-text-color, #555555);">Total Recursive Calls</td>
                    <td style="padding: 8px; font-weight: bold; color: var(--colab-primary-text-color, #212529);">{stats['calls']:,}</td>
                </tr>
                <tr style="border-bottom: 1px solid var(--colab-border-color, #ddd);">
                    <td style="padding: 8px; color: var(--colab-secondary-text-color, #555555);">Peak Stack Depth</td>
                    <td style="padding: 8px; font-weight: bold; color: var(--colab-primary-text-color, #212529);">{stats['max_depth']:,}</td>
                </tr>
                <tr>
                    <td style="padding: 8px; color: var(--colab-secondary-text-color, #555555);">Elapsed Time</td>
                    <td style="padding: 8px; font-weight: bold; color: #28a745;">{stats['time_seconds']:.6f} seconds</td>
                </tr>
            </table>
        </div>
        """
        display(HTML(metrics_html))
        display(HTML("<br>"))
        display(HTML(generate_trace_explanation(m_val, n_val)))


calculate_btn.on_click(on_click_calculate)

layout = widgets.VBox(
    [
        widgets.HTML(
            "<h3 style='color: var(--colab-primary-text-color, #212529);'>Ackermann Interactive Control Panel</h3>"
        ),
        m_slider,
        n_slider,
        calculate_btn,
        output_area,
    ]
)

display(layout)

# Pre-populate the panel with the default A(3, 4) analysis
on_click_calculate(None)


## 5. Demystifying the Ackermann Function: Why Does It Explode?

A helpful way to understand $A(m, n)$ is to view $m$ as selecting a progressively stronger hyper-operation, while $n$ acts as the operand fed into that tier.

### How Each Level of $m$ Behaves:
* **$m = 0$ (Successor / Increment):**
  $$A(0, n) = n + 1$$
  Advances $n$ by a single unit, producing strictly linear growth.

* **$m = 1$ (Addition Tier):**
  $$A(1, n) = n + 2$$
  Unwinds into repeated increments of $1$, behaving like basic addition.

* **$m = 2$ (Multiplication Tier):**
  $$A(2, n) = 2n + 3$$
  Applies repeated doubling-style steps (for example, $A(2, 5) = 13$ and $A(2, 6) = 15$).

* **$m = 3$ (Exponentiation / Power Tier):**
  $$A(3, n) = 2^{(n+3)} - 3$$
  Growth now scales exponentially in $n$:
  * $A(3, 4) = 2^7 - 3 = 125$ (`10,307` recursive calls)
  * $A(3, 5) = 2^8 - 3 = 253$ (`42,438` recursive calls)
  * $A(3, 6) = 2^9 - 3 = 509$ (`172,233` recursive calls)

* **$m = 4$ (Tetration / Power-Tower Tier):**
  $$A(4, n) = \underbrace{2^{2^{.^{.^{2}}}}}_{n + 3 \text{ times}} - 3$$
  Here the exponent itself becomes a tower of powers of $2$:
  * $A(4, 0) = A(3, 1) = 13$ (still tractable in `107` calls)
  * $A(4, 1) = A(3, 13) = 2^{16} - 3 = 65,533$
  * $A(4, 2) = A(3, 65533) = 2^{65536} - 3$

> **Scale Check:** Writing $2^{65536}$ in base 10 requires **19,729 digits**. Evaluating $A(4, 2)$ directly through naive recursion is physically impossible within normal memory and time limits.

## 6. What Does This Have to Do with Computability?

1. **Bounded-Loop (Primitive Recursive) Programs:** Standard arithmetic, factorials, and fixed-depth exponentiation can all be written using `for` loops whose iteration limits are known before each loop begins.
2. **Why Ackermann Breaks Fixed Nesting:** Every fixed nesting depth of `for` loops corresponds to a bounded tier of growth, whereas $A(m, n)$ climbs through *all* tiers as $m$ increases—eventually outgrowing any primitive recursive function.
3. **Computable Without Being Primitive Recursive:** Because the three defining rules always shrink $(m, n)$ lexicographically and eventually terminate for every non-negative integer pair, $A(m, n)$ is genuinely **computable**—proving that general recursion (`while` loops / unbounded call stacks) is strictly more powerful than bounded iteration alone.
